# 📗 DeepEval: 지표와 테스트 케이스

## 오늘 배울 내용

교안 01에서는 판사 프롬프트를 직접 만들어 답변을 채점하고 사람 판정과 대조했습니다. 이번에는 같은 평가를 **DeepEval**로 옮겨, 평가 사례와 지표를 코드로 정리하고 여러 사례를 한 번에 실행합니다.

<img src="./images/00_lesson02_goals.png" width="720" alt="오늘의 학습 목표 네 가지: 평가 사례 만들기, 기준에 따라 채점하기, 지표를 나눠 보기, 여러 사례를 함께 평가하기">

**학습 목표:** 루나 답변으로 다음을 직접 해 봅니다.

| 학습 목표 | 절 |
|---|---|
| ① 질문·평가할 답변·기준 답변·근거를 `LLMTestCase` 한 건에 담기 | 3절 |
| ② G-Eval 기준을 판단 단계와 rubric으로 적고, 기준 답변을 넣었을 때와 뺐을 때의 판정 비교하기 | 4절 |
| ③ 근거와의 모순·무례한 표현·집단 고정관념을 전용 지표(Hallucination·Toxicity·Bias)로 평가하기 | 5절, 6-1 |
| ④ 평가 축마다 알맞은 방법으로 판정하고, 사람 판정과 대조해 판사 성적표 만들기 | 6절 |
| ⑤ 새 답변을 만들어 `evaluate()`로 여러 지표를 함께 적용하기 | 7절 |
| ⑥ 함께 따라하기: 굿즈 스토어 상담 답변을 정책 기준으로 평가하기 | 8절 |

### LangChain과 DeepEval의 역할

LangChain은 챗봇 답변을 만들고(7·8절), DeepEval은 판사 모델로 그 답변을 평가합니다. 검색기는 실행하지 않고 미리 준비한 근거로 평가하며, 같은 방법을 RAG 답변에도 쓸 수 있습니다. [DeepEval 공식 지표 소개](https://deepeval.com/docs/metrics-introduction)

## 1. DeepEval로 반복 평가하기

**DeepEval**은 답변 평가를 파이썬 코드로 정리해 반복 실행하는 평가 도구입니다. 평가 기준은 사람이 정하고, 판사 모델이 그 기준으로 답변을 판정합니다. 사례와 지표를 코드로 남겨 두면 모델이나 프롬프트를 바꿀 때 같은 평가를 다시 돌릴 수 있습니다.

<img src="./images/11_framework.png" width="760" alt="평가 사례에 질문·답·근거를 담고, 지표를 골라 evaluate로 실행한 뒤 케이스별 결과를 확인하는 흐름">

평가는 아래 네 단계로 진행하며, 3–7절에서 차례로 실습합니다.

| 순서 | 하는 일 | DeepEval 코드 |
|---|---|---|
| 1 | 질문·평가할 답변·근거를 **평가 사례** 한 건으로 묶기 | `LLMTestCase` |
| 2 | 판정 기준이나 목적에 맞는 **지표** 고르기 | `GEval`, `HallucinationMetric` 등 |
| 3 | 사례에 지표 적용하기 | 한 건: `지표.measure(사례)` / 여러 건: `evaluate(사례 목록, 지표 목록)` |
| 4 | 점수·통과 여부·판정 이유 읽기 | `score`, `success`, `reason` |

## 2. 실습 준비

실습 대상을 소개하고, 답변 모델·판사 모델을 만든 뒤 실습 자료를 읽습니다.

### 2-1. 실습 대상: 루나와 루나 굿즈 스토어

<table>
<tr>
<td width="250" valign="top" align="center">
<img src="./images/01_luna_character.png" width="230" alt="별 장식과 분홍 리본, 흰색·분홍색 무대 의상과 마이크를 든 루나">
<br><sub>수업용 가상 캐릭터 · 루나</sub>
</td>
<td valign="top">

**루나**는 팬과 대화하는 **수업용 버추얼 아이돌 챗봇**입니다. 교안 01과 같은 설정을 씁니다.

- 따뜻하고 경쾌한 말투로 팬을 응원합니다.
- 플레이리스트 질문과 가상 팬미팅 안내에 답합니다.
- 설정과 공지는 `data/luna_guide.json`의 L1–L8 문장에 있습니다.

**루나 굿즈 스토어 상담 챗봇**(8절)은 응원봉·포토카드 같은 가상 굿즈의 교환·반품 질문에 정책(P1–P5)대로 답합니다.

등장하는 인물·설정·일정·정책은 모두 실습용 가상 자료입니다.

</td>
</tr>
</table>

답변은 아래 **평가 축**(답변을 나눠 보는 품질 기준)으로 평가합니다. 축마다 알맞은 판정 방법은 6절에서 고릅니다.

| 평가 축 | 확인할 점 |
|---|---|
| 페르소나 일관성 | 루나의 설정과 말투를 지키는가 |
| 질문 충족도·근거성 | 질문에 답하고, 공지와 맞으며 없는 정보를 만들지 않는가 |
| 형식 준수 | 지정한 줄 수와 항목 순서를 지키는가 |
| 존중 표현·편향 | 팬을 모욕하지 않고, 성별로 취향을 단정하지 않는가 |
| 정책 준수(8절) | 정책 조건을 빠짐없이 안내하고 없는 약속을 하지 않는가 |

### 2-2. 답변 모델과 판사 모델 설정

| 코드 이름 | 역할 | 객체 |
|---|---|---|
| `answer_llm` | 챗봇 답변을 만듭니다. 7·8절에서 새 답변을 만들 때만 씁니다. | LangChain `ChatOpenAI` |
| `judge_llm` | 답변을 평가합니다. DeepEval 지표에 `model=judge_llm`으로 넘깁니다. | DeepEval `GPTModel` |

답변 모델과 판사 모델을 나누면 **자기 선호**(판사가 자기와 같은 모델이 만든 답을 더 좋게 보는 경향)를 줄일 수 있습니다. 판사의 점수와 판정 이유는 실행마다 달라질 수 있으므로, 결과가 **확인할 점**의 기대 결과와 다르면 판정 이유를 읽고 원인을 찾습니다.

**`logprobs`:** 모델이 다음 토큰(글자·단어 조각)을 고를 때 후보마다 매긴 확률입니다. DeepEval 4.1.1의 G-Eval은 판사가 고른 점수 숫자의 `logprobs`로 점수를 가중 평균합니다. 기본 판사 `gpt-4.1`은 이를 지원하며, 다른 판사 모델로 바꾸면 지원 여부를 확인합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

pd.set_option("display.max_colwidth", None)  # 판정 이유 같은 긴 문장을 자르지 않고 보여 줍니다.

# DeepEval이 사용 기록을 외부로 보내지 않게 합니다. deepeval을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"
import deepeval
from deepeval.models import GPTModel

# material_dir는 data가 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 .env에서 채웁니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
answer_model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
answer_llm = ChatOpenAI(model=answer_model_name, use_responses_api=True)

# 판사 모델: 답변 모델과 다른 모델(OPENAI_EVAL_MODEL)로 채점합니다.
judge_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-4.1")
# temperature=0은 출력 변동을 낮추지만 같은 판정을 보장하지는 않습니다.
judge_llm = GPTModel(model=judge_model_name, temperature=0)

print("답변 모델:", answer_model_name)
print("판사 모델:", judge_model_name)
print("DeepEval 버전:", deepeval.__version__)


# [보조 함수] JSON 자료 읽기
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))

### 2-3. 실습 자료 살펴보기

이 교안에서 쓰는 자료는 아래와 같습니다. 이 절에서는 루나 설정과, 3–5절에서 쓸 연습 답변 8건을 읽습니다.

| 파일 | 들어 있는 내용 | 쓰는 곳 |
|---|---|---|
| `luna_guide.json` | 루나 설정과 가상 팬미팅 공지 문장(L1–L8), 평가에 쓸 정보의 범위 | 전체 |
| `luna_metric_cases.json` | 주된 결함(`defect`)을 하나씩 심은 루나 답변 8건 | 3–5절 |
| `luna_pairwise_cases.json` | 두 답 비교 사례 6건(이 교안은 LP04만 사용) | 4-4 |
| `luna_eval_cases.json` | 평가 축별 루나 답변 15건과 사람 판정 | 6절 |
| `luna_coverage_cases.json` | 평가 범위표에서 고른 새 질문 6건 | 7절 |
| `store_*.json` | 굿즈 스토어 가상 정책 5개, 사람 판정이 있는 상담 기록 8건(답 16개), 새 질문 6건 | 8절 |

In [ ]:
# 루나 설정과 가상 공지 문장(L1–L8)을 ID로 찾을 수 있게 보관합니다.
guide = read_json("luna_guide.json")
sentence_by_id = {sentence["id"]: sentence["text"] for sentence in guide["sentences"]}


# [보조 함수] 근거 ID → 공지 문장
def luna_context(sentence_ids):
    """근거 ID 목록(예: ["L4", "L5"])을 공지 문장 목록으로 반환합니다."""
    return [sentence_by_id[sentence_id] for sentence_id in sentence_ids]


for sentence_id, text in sentence_by_id.items():
    print(f"{sentence_id}: {text}")
print("\n자료 범위:", guide["evidence_scope"])

In [ ]:
# 주된 결함을 하나씩 심은 루나 답변 8건을 읽습니다. defect_by_id는 결과 표에 결함을 붙일 때 씁니다.
luna_metric_rows = read_json("luna_metric_cases.json")
defect_by_id = {case["id"]: case["defect"] for case in luna_metric_rows}

display(pd.DataFrame([
    {"케이스 ID": case["id"], "주 결함": case["defect"], "질문": case["input"],
     "답변": case["actual_output"], "기준 답변": case["expected_output"],
     "근거 ID": ", ".join(case["context_ids"])}
    for case in luna_metric_rows
]))

## 3. 평가 사례 한 건을 담는 `LLMTestCase`

**평가 사례**는 질문·평가할 답변·기준 답변·근거를 한데 모은 자료입니다. DeepEval에서는 사례 한 건을 `LLMTestCase`로 만듭니다. 이 절에서는 2-3의 루나 답변 8건을 `LLMTestCase`로 바꿉니다.

<img src="./images/12_testcase_fields.png" width="760" alt="LLMTestCase: input은 질문, actual_output은 평가할 답변, expected_output은 사람이 확인한 기준 답변, context는 미리 준비한 근거 문장, retrieval_context는 검색기가 실제 찾아온 문서입니다. 이번 실습에서는 검색기를 실행하지 않습니다.">

| 필드 | 담는 내용 | 이 교안에서 |
|---|---|---|
| `input` | 사용자의 질문 | 팬의 질문 |
| `actual_output` | 평가할 답변 | 루나의 답변 |
| `expected_output` | 사람이 확인한 **기준 답변**. 지표에 따라 생략 가능 | 모범 답 문장(예: `event_ok`) 또는 기대 행동 설명(예: `ok`의 "따뜻하고 경쾌한 루나의 말투로 응원한다.") |
| `context` | 평가 전에 **미리 준비한** 근거 문장 목록 | 공지 문장(L1–L8) 중 질문에 맞는 것. G-Eval·Hallucination이 사실을 대조할 때 읽음 |
| `retrieval_context` | RAG 검색기가 **실제로 찾아온** 문서 목록 | 검색기를 실행하지 않으므로 비워 둠 |
| `name` | 사례 이름 | 케이스 ID. 결과를 사례·사람 판정과 연결할 때 씀 |

정답이 하나로 정해지지 않는 응원·페르소나 질문에는 기대 행동 설명이 더 알맞습니다.

In [ ]:
from deepeval.test_case import LLMTestCase

# 루나 답변 8건을 LLMTestCase로 만듭니다.
luna_test_cases = []
for case in luna_metric_rows:
    luna_test_cases.append(LLMTestCase(
        input=case["input"],
        actual_output=case["actual_output"],
        expected_output=case["expected_output"],
        context=luna_context(case["context_ids"]),  # 근거 ID를 공지 문장 목록으로 바꿔 넣습니다.
        name=case["id"],
    ))
test_case_by_id = {test_case.name: test_case for test_case in luna_test_cases}

# 시작 시각을 잘못 안내한 사례의 필드를 확인합니다.
wrong_case = test_case_by_id["event_wrong"]
print("input:", wrong_case.input)
print("actual_output:", wrong_case.actual_output)
print("expected_output:", wrong_case.expected_output)
print("context:", wrong_case.context)

**확인할 점:** `actual_output`은 오후 8시, `expected_output`은 오후 7시로 다릅니다. `expected_output`은 기대하는 답의 예시, `context`는 사실을 대조할 근거 문장의 **목록**입니다. 4절에서 판사가 이 차이를 찾아내는지 봅니다.

## 4. G-Eval로 평가 기준을 구체화하기

**G-Eval**은 사람이 정한 기준을 판단 단계로 나누고, 판사 모델이 그 단계에 따라 점수와 판정 이유를 내게 하는 지표입니다. 이 절에서는 같은 루나 답변 8건으로 판단 단계(4-1), 기준 답변(4-2), rubric(4-3), 두 답 비교(4-4)를 차례로 바꿔 봅니다. [DeepEval G-Eval 공식 문서](https://deepeval.com/docs/metrics-llm-evals)

### 4-1. 판사 모델이 따를 단계를 정하기

<img src="./images/13_geval_flow.png" width="760" alt="질문·답변·근거를 입력으로, 평가 기준과 판단 단계를 정해 판사 모델이 점수와 판정 이유를 내는 흐름">

루나의 팬미팅 안내라면 질문에 답했는지, 일정이 공지와 맞는지, 공지에 없는 정보를 지어내지 않았는지를 단계로 적습니다.

| 설정 | 뜻 | 이 예에서 |
|---|---|---|
| `name` | 지표 이름 | `질문 충족과 근거성` |
| `evaluation_steps` | 판사가 따를 판단 순서. 직접 쓰면 순서가 고정되고, 생략하고 `criteria`(기준 문장)만 주면 판사가 단계를 만듦 | 직접 적음: 질문 → 근거 → 지어낸 정보 |
| `evaluation_params` | 판사에게 보여 줄 `LLMTestCase` 필드. `SingleTurnParams`로 고름 | `INPUT`·`ACTUAL_OUTPUT`·`CONTEXT` |
| `model` | 판사 모델 | `judge_llm` |
| `threshold` | **통과 기준값** | 0.7. 실습용 출발값이며 사람 판정과 비교해 조정 |

G-Eval 점수는 0–1이며 높을수록 좋습니다. 점수가 `threshold` 이상이면 통과(`success=True`)입니다.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# 질문 충족과 근거성을 판단 단계로 적습니다. 말투는 이 지표에서 보지 않습니다.
accuracy_steps = [
    "input에서 사용자가 묻는 핵심을 확인한다.",
    "actual_output이 input의 질문에 직접 답했는지 확인한다.",
    "actual_output의 일정·사실이 context와 일치하는지 확인한다.",
    "context가 뒷받침하지 않는 구체 사실을 단정하면 감점한다.",
    "말투와 길이는 이 지표의 점수에 반영하지 않는다.",
]

# expected_output 없이 질문·답변·근거만 판사에게 보여 줍니다.
accuracy_metric = GEval(
    name="질문 충족과 근거성",
    evaluation_steps=accuracy_steps,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.7,
)

In [ ]:
# [보조 함수] 지표 하나로 사례를 한 건씩 채점합니다(4-2에서도 씁니다).
def measure_each(metric, test_cases):
    """measure()로 사례마다 채점해 케이스 ID·주 결함·점수·통과·판정 이유 표를 반환합니다."""
    rows = []
    for test_case in test_cases:
        metric.measure(test_case)  # 판사 호출. 결과(score·success·reason)는 지표 객체에 남습니다.
        rows.append({
            "케이스 ID": test_case.name,
            "주 결함": defect_by_id[test_case.name],
            "점수": round(metric.score, 2),
            "통과": metric.success,
            "판정 이유": metric.reason,
        })
    return pd.DataFrame(rows)


# 루나 답변 8건을 4-1의 지표로 채점합니다(판사 요청 8개).
accuracy_table = measure_each(accuracy_metric, luna_test_cases)

In [ ]:
# 점수·통과를 주 결함과 나란히 봅니다.
display(accuracy_table)

**확인할 점:** `점수`·`통과` 열을 `주 결함` 열과 비교합니다.

| 사례 | 기대 결과 | 까닭 |
|---|---|---|
| `under_answer`, `event_wrong`, `unknown_detail` | 0.7 미만, `통과` False | 이 지표가 겨냥한 결함(질문 미충족, 틀린 시각, 공지에 없는 정보) |
| `ok`, `event_ok` | 0.7 이상, `통과` True | 결함 없음 |
| `rude`, `stereotype`, `persona_drift` | 정해지지 않음 | 모욕·고정관념·말투는 판단 단계에 없음. `persona_drift`는 L1과 모순되어 근거 확인에서 잡힐 수는 있음 |

기대와 다른 줄은 `판정 이유`로 판사가 무엇을 놓쳤는지 읽습니다. 지표는 겨냥하지 않은 결함을 놓치거나 우연히만 잡으므로, 5–6절에서 목적에 맞는 지표를 따로 씁니다.

### 4-2. 기준 답변을 함께 보여 주기

`evaluation_params`에 `expected_output`(기준 답변)을 더하면 판사가 비교할 예시를 얻습니다. 같은 8건을 기준 답변과 함께 다시 채점해 4-1의 점수와 나란히 봅니다. 표현이 달라도 같은 정보를 전달하면 통과해야 하므로, 판단 단계에 "문장을 그대로 복사하지 않아도 된다"를 넣습니다.

In [ ]:
# 4-1의 단계에 expected_output(기준 답변) 비교를 더한 G-Eval입니다. 6절과 7절에서도 다시 씁니다.
accuracy_ref_metric = GEval(
    name="질문 충족과 근거성(기준 답변 포함)",
    evaluation_steps=[
        "input에서 사용자가 묻는 핵심을 확인한다.",
        "actual_output이 질문에 답하고 필요한 정보를 포함하는지 살핀다.",
        "actual_output의 핵심 내용이 expected_output 및 context와 맞는지 확인한다.",
        "context가 뒷받침하지 않는 구체 사실을 단정하면 감점한다.",
        "문장을 그대로 복사하지 않아도 같은 기준을 충족하면 통과로 본다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.7,
)

In [ ]:
# 같은 8건을 기준 답변과 함께 다시 채점합니다(판사 요청 8개).
accuracy_ref_table = measure_each(accuracy_ref_metric, luna_test_cases)

In [ ]:
# 4-1의 점수(기준 답변 없이)와 나란히 둡니다. 두 표는 같은 사례 순서입니다.
compare_table = accuracy_table[["케이스 ID", "주 결함", "점수"]].rename(columns={"점수": "기준 답변 없이(4-1)"})
compare_table["기준 답변 포함"] = accuracy_ref_table["점수"]
compare_table["기준 답변 포함 이유"] = accuracy_ref_table["판정 이유"]
display(compare_table)

**확인할 점:** `기준 답변 없이(4-1)`과 `기준 답변 포함`을 나란히 봅니다.

- 기대 결과: 두 열 모두 `under_answer`·`event_wrong`·`unknown_detail`은 0.7 미만, `ok`·`event_ok`는 0.7 이상입니다.
- 0.7을 사이에 두고 갈린 줄(한쪽만 통과)은 `기준 답변 포함 이유`를 읽습니다. 모범 답 문장인 `event_wrong`은 시각 차이(오후 8시 대 7시)를, 기대 행동 설명인 `ok`·`under_answer`는 그 행동을 근거로 짚어야 합니다.
- 기준 답변을 줘도 오판은 남고, 기준 답변이 틀리면 판정도 틀어지므로 기준 답변은 사람이 먼저 검토합니다.

### 4-3. rubric으로 점수 구간 정하기

**rubric**(점수별 기준, 교안 01 3-2)은 점수 구간마다 어떤 답변인지 미리 적은 것입니다. 구간의 뜻을 정해 주면 판사가 같은 잣대로 점수를 매기기 쉬워집니다. 구간은 서로 겹치지 않게 정하고, 아래 표의 문장을 그대로 판사에게 줍니다.

| 점수 구간 | 기대 결과(`expected_outcome`) |
|---|---|
| 0–3 | 핵심 안내가 공지와 모순된다. |
| 4–7 | 핵심 안내는 맞지만 확인되지 않은 구체 사실이 있다. |
| 8–10 | 질문에 답하고 공지 범위 안에서 안내한다. |

rubric은 0–10점으로 적지만 결과 `score`는 0–1로 바뀝니다(예: 7점 → 0.7). 8–10 구간만 통과로 보도록 `threshold`를 0.8로 두고, 공지에 없는 VIP 입장 시간을 단정한 `unknown_detail`을 채점합니다.

In [ ]:
from deepeval.metrics.g_eval import Rubric

# 4-1과 같은 단계에 점수 구간(rubric)을 더합니다. 8–10점만 통과로 보도록 threshold를 0.8로 둡니다.
accuracy_rubric_metric = GEval(
    name="질문 충족과 근거성(rubric)",
    evaluation_steps=accuracy_steps,
    rubric=[
        Rubric(score_range=(0, 3), expected_outcome="핵심 안내가 공지와 모순된다."),
        Rubric(score_range=(4, 7), expected_outcome="핵심 안내는 맞지만 확인되지 않은 구체 사실이 있다."),
        Rubric(score_range=(8, 10), expected_outcome="질문에 답하고 공지 범위 안에서 안내한다."),
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
    threshold=0.8,
)

accuracy_rubric_metric.measure(test_case_by_id["unknown_detail"])  # 판사 호출
print("점수(0–1):", round(accuracy_rubric_metric.score, 2))
print("통과:", accuracy_rubric_metric.success)
print("판정 이유:", accuracy_rubric_metric.reason)

**확인할 점:** 답의 핵심인 VIP 입장 시각을 L5("VIP 입장 시간과 공연 종료 시각은 제공된 공지에 없다")가 공지에 없다고 하므로, 기대 결과는 0–3 구간(0.3 이하)과 `통과` False입니다. `판정 이유`가 L5를 짚는지 확인합니다. 점수가 `threshold`에 가까우면 실행마다 통과 표시가 바뀔 수 있습니다.

### 4-4. 두 답 비교(Pairwise)를 `ArenaGEval`로

`ArenaGEval`은 같은 질문에 대한 후보 답변들을 판단 단계에 따라 비교해 승자를 고르는 지표입니다. LP04의 두 답(시작 시각이 틀린 답, 공지와 맞는 답)을 같은 질문·기준 답변·근거로 비교합니다.

| 비교 방식 | 특징 |
|---|---|
| 직접 만든 Pairwise(교안 01) | 무승부를 허용하고, 순서를 바꿔 두 번 물어 위치 효과(교안 01 5-2)를 직접 확인 |
| `ArenaGEval` | 승자 이름만 돌려주며 무승부가 없음. DeepEval 4.1.1은 후보 이름을 가짜 이름으로 가리고 제시 순서를 무작위로 섞어 판사에게 보여 줌 |

이름 가리기와 순서 섞기는 이름·위치 효과를 줄이지만, 한 번의 결과를 확정 판단으로 쓰지는 않습니다. 같은 비교를 두 번 실행해 승자가 유지되는지 봅니다. [DeepEval 4.1.1 ArenaGEval 구현](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/arena_g_eval/utils.py)

In [ ]:
from deepeval.metrics import ArenaGEval
from deepeval.test_case import ArenaTestCase, Contestant

# LP04: 같은 질문에 대한 두 후보를 가져옵니다. 기준 답변은 3절 event_ok와 같은 문장입니다.
arena_source = next(row for row in read_json("luna_pairwise_cases.json") if row["pair_id"] == "LP04")
arena_context = luna_context(arena_source["context_ids"])
arena_reference = test_case_by_id["event_ok"].expected_output

# 후보마다 질문·기준 답변·근거를 똑같이 두고 답변만 다르게 합니다.
arena_case = ArenaTestCase(contestants=[
    Contestant(name="시간 오류 답", test_case=LLMTestCase(
        input=arena_source["question"], actual_output=arena_source["answer_a"],
        expected_output=arena_reference, context=arena_context)),
    Contestant(name="공지와 맞는 답", test_case=LLMTestCase(
        input=arena_source["question"], actual_output=arena_source["answer_b"],
        expected_output=arena_reference, context=arena_context)),
])

# 기준 답변·공지와 맞는 정도로 후보를 비교합니다.
arena_metric = ArenaGEval(
    name="팬미팅 안내 비교",
    evaluation_steps=[
        "각 후보가 같은 질문에 답했는지 확인한다.",
        "답변의 날짜와 시간이 expected_output 및 context와 일치하는지 비교한다.",
        "더 정확하고 근거에 맞는 후보를 고른다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    model=judge_llm,
)

In [ ]:
# 같은 비교를 두 번 실행합니다. 실행마다 후보 이름이 가려지고 제시 순서가 무작위로 섞입니다.
for run in [1, 2]:
    arena_metric.measure(arena_case)  # 판사 호출
    print(f"{run}회차 승자:", arena_metric.winner)
    print("  판정 이유:", arena_metric.reason)

**확인할 점:** 두 회차 모두 `공지와 맞는 답`이 이기고 `판정 이유`가 공지의 시작 시각(오후 7시)을 짚는지 봅니다. 사람 선호(`human_winner`)도 B, 즉 공지와 맞는 답입니다. 두 회차의 승자가 다르면 판정이 제시 순서나 호출 변동에 흔들린 것이므로 확정 판단으로 쓰지 않습니다.

## 5. 답변과 근거의 모순 확인하기: Hallucination

4절의 G-Eval은 기준을 직접 적는 지표였습니다. 이 절에서는 목적이 정해진 **전용 지표** `HallucinationMetric`으로 답변이 준비한 근거(`context`)와 모순되는지 확인합니다.

<img src="./images/14_hallucination_check.png" width="760" alt="답변을 미리 준비한 근거 문장과 하나씩 비교해 모순을 찾고, 모순 판정 수를 전체 판정 수로 나눈 점수는 낮을수록 좋다">

| 항목 | DeepEval 4.1.1의 동작 |
|---|---|
| 판정 | 판사가 `context`의 문장마다 `yes`(답변이 그 문장과 모순되지 않음) 또는 `no`(모순)를 매김. 정보가 덜 담긴 것은 모순으로 보지 않음 |
| 점수 | `context` 문장 중 `no` 판정을 받은 비율. 예: 2문장 중 1개가 `no`면 0.5. **낮을수록 좋음** |
| 통과 | 점수가 `threshold`(0.5) **이하**이면 통과. G-Eval(이상이면 통과)과 방향이 반대 |
| 한계 | 모순만 찾으므로, 공지에 없는 정보를 지어내도 어떤 문장과도 모순되지 않으면 통과할 수 있음 |

[DeepEval 4.1.1 Hallucination 구현](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/hallucination/hallucination.py)

### 5-1. 공지 문장별로 모순 판정하기

정상 안내(`event_ok`), 틀린 시각(`event_wrong`), 공지에 없는 VIP 입장 시간(`unknown_detail`)을 채점합니다.

In [ ]:
from deepeval.metrics import HallucinationMetric

hallucination_metric = HallucinationMetric(model=judge_llm, threshold=0.5)

# 세 사례를 채점합니다(판사 요청 3개). 다음 사례를 채점하면 결과가 덮어써지므로 사례마다 꺼내 둡니다.
hallucination_rows = []
verdicts_by_id = {}
for case_id in ["event_ok", "event_wrong", "unknown_detail"]:
    hallucination_metric.measure(test_case_by_id[case_id])  # 판사 호출
    verdicts_by_id[case_id] = hallucination_metric.verdicts  # 문장별 판정
    hallucination_rows.append({
        "케이스 ID": case_id,
        "주 결함": defect_by_id[case_id],
        "점수(낮을수록 좋음)": round(hallucination_metric.score, 2),
        "통과": hallucination_metric.success,
    })

In [ ]:
# 사례별 점수를 보고, 문장별 판정을 근거 문장과 나란히 펼칩니다(두 목록은 같은 순서).
display(pd.DataFrame(hallucination_rows))
verdict_rows = []
for case_id, verdicts in verdicts_by_id.items():
    for sentence, verdict in zip(test_case_by_id[case_id].context, verdicts):
        verdict_rows.append({"케이스 ID": case_id, "공지 문장": sentence,
                             "판정(yes=모순 없음, no=모순)": verdict.verdict, "이유": verdict.reason})
display(pd.DataFrame(verdict_rows))

**확인할 점:**

| 사례 | 기대 결과 | 의미 |
|---|---|---|
| `event_ok` | L4 `yes`, 점수 0, `통과` True | 정상 안내는 공지와 모순이 없음 |
| `event_wrong` | L4 `no`, 점수 1, `통과` False | 공지 시각(오후 7시)과의 모순을 찾음 |
| `unknown_detail` | L4 `yes`, L5 `no`, 점수 0.5, `통과` True | L5("VIP 입장 시간과 공연 종료 시각은 제공된 공지에 없다")가 있어서 지어낸 VIP 시각을 모순으로 잡음. L5가 `yes`면 판사가 놓친 것 |

`unknown_detail`은 L5를 `no`로 잡아도 점수가 0.5(2문장 중 1개)라서 `threshold` 0.5 이하, 즉 **통과**로 표시됩니다. 점수만 보지 말고 문장별 판정을 읽어야 하는 이유입니다.

### 5-2. 질문과 무관한 근거까지 넣으면

`event_ok`의 답변을 관련 공지(L4)만 줄 때와 공지 전체(L1–L8)를 줄 때로 나눠 채점합니다. `context`를 어떻게 고르느냐에 따라 판정이 달라지는지 봅니다.

In [ ]:
# 같은 답변을 관련 공지(L4)만 준 사례와 공지 전체(L1–L8)를 준 사례로 채점합니다(판사 요청 2개).
event_case = test_case_by_id["event_ok"]
event_all_case = LLMTestCase(
    input=event_case.input,
    actual_output=event_case.actual_output,
    context=list(sentence_by_id.values()),  # 질문과 무관한 문장까지 모두 넣습니다.
)

hallucination_metric.measure(event_case)  # 판사 호출
related_score = hallucination_metric.score
hallucination_metric.measure(event_all_case)  # 판사 호출
all_score, all_verdicts = hallucination_metric.score, hallucination_metric.verdicts

In [ ]:
print("관련 공지(L4)만 줬을 때 점수:", round(related_score, 2))
print("공지 전체(L1–L8)를 줬을 때 점수:", round(all_score, 2))

# 공지 전체를 줬을 때의 문장별 판정을 펼칩니다.
display(pd.DataFrame([
    {"공지 문장": sentence, "판정(yes=모순 없음, no=모순)": verdict.verdict, "이유": verdict.reason}
    for sentence, verdict in zip(event_all_case.context, all_verdicts)
]))

**확인할 점:** 기대 결과는 두 경우 모두 점수 0입니다. 공지 전체를 줬을 때 점수가 0보다 크면 `no`가 붙은 문장을 찾습니다. 답변이 언급하지 않은 문장(예: L6 성별, L7 생활 정보)에 `no`가 붙었다면 판사가 질문과 무관한 문장 때문에 모순을 찾은 것입니다. 근거는 많을수록 좋은 것이 아니라 질문과 관련 있어야 합니다.

## 6. 평가 축별로 판정하고 사람 판정과 대조하기

3–5절에서는 결함을 하나씩 심은 연습 사례로 지표의 동작을 봤습니다. 이 절에서는 **사람 판정이 있는 루나 답변 15건**을 평가 축마다 알맞은 방법으로 판정하고, 판사 판정을 사람 판정과 대조해 **판사 성적표**를 만듭니다.

| 하위 절 | 평가 축 | 사례 | 판정 방법 |
|---|---|---|---|
| 6-1 | 존중 표현·편향 | L12–L15 | `ToxicityMetric`·`BiasMetric` |
| 6-2 | 페르소나 일관성 | L01–L03 | G-Eval(루나 페르소나) |
| 6-3 | 질문 충족도·근거성 | L04–L06, L09–L11 | G-Eval(4-2의 `accuracy_ref_metric`) |
| 6-4 | 형식 준수 | L07·L08 | 코드 검사 |
| 6-5 | 15건 전체 | L01–L15 | 판사 성적표 |

In [ ]:
# 사람 판정이 있는 루나 평가 사례 15건을 읽고, 케이스 ID로 사람 판정을 찾을 수 있게 합니다.
luna_eval_rows = read_json("luna_eval_cases.json")
human_label_by_id = {row["case_id"]: row["human_label"] for row in luna_eval_rows}


# [보조 함수] 평가 축으로 사례 고르기
def luna_eval_cases_for(dimensions):
    """평가 축(dimension) 목록에 속한 사례를 LLMTestCase 목록으로 반환합니다. name에는 케이스 ID를 넣습니다."""
    return [
        LLMTestCase(input=row["input"], actual_output=row["actual_output"],
                    expected_output=row["expected_output"], context=row["context"], name=row["case_id"])
        for row in luna_eval_rows if row["dimension"] in dimensions
    ]


display(pd.DataFrame([
    {"케이스 ID": row["case_id"], "평가 축": row["dimension"], "사람 판정": row["human_label"],
     "질문": row["input"], "답변": row["actual_output"], "사람이 붙인 이유": row["reason"]}
    for row in luna_eval_rows
]))

**여러 사례는 `evaluate()`로 실행합니다.** 4–5절의 `measure()`는 사례 한 건씩 채점했습니다. `evaluate()`는 사례 목록에 지표 목록을 적용하고, 판사 호출을 동시에 보내 결과를 한꺼번에 돌려줍니다.

| 설정·결과 | 뜻 |
|---|---|
| `AsyncConfig(max_concurrent=4)` | 동시에 처리할 사례 수의 상한. 기다리는 시간은 줄지만 요청 수와 비용은 그대로 |
| `DisplayConfig(print_results=False, show_indicator=False)` | DeepEval의 긴 결과 출력과 진행 표시를 끄고 표로 직접 봄 |
| 결과의 `test_results` | 사례마다 하나씩. 각 사례의 `metrics_data`에 지표별 `score`·`success`·`reason`이 들어 있음 |

결과 순서는 입력 순서와 다를 수 있으므로 사례 이름(`name`)으로 사람 판정과 연결합니다. 아래 보조 함수 세 개(`run_metrics`: 판사 실행, `add_human_labels`: 사람 판정 붙이기, `show_judge_table`: 교안 01과 같은 결과 보기)를 8절까지 계속 씁니다.

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig


# [보조 함수] 판사 실행: evaluate() 결과를 사례·지표마다 한 행인 표로 바꿉니다.
def run_metrics(test_cases, metrics):
    """test_cases에 metrics를 모두 적용해 케이스 ID·지표·판사 판정·점수·판정 이유 열의 DataFrame을 반환합니다."""
    result = evaluate(
        test_cases=test_cases,
        metrics=metrics,
        async_config=AsyncConfig(max_concurrent=4),
        display_config=DisplayConfig(print_results=False, show_indicator=False),
    )
    rows = []
    for test_result in result.test_results:  # 사례 하나
        for metric_data in test_result.metrics_data:  # 그 사례에 적용한 지표 하나
            rows.append({
                "케이스 ID": test_result.name,
                "지표": metric_data.name,
                # success는 지표의 점수 방향(높을수록/낮을수록 좋음)을 이미 반영한 통과 여부입니다.
                "판사 판정": "pass" if metric_data.success else "fail",
                "점수": round(metric_data.score, 2),
                "판정 이유": metric_data.reason,
            })
    return pd.DataFrame(rows).sort_values(["케이스 ID", "지표"], ignore_index=True)


# [보조 함수] 사람 판정 붙이기
def add_human_labels(judge_table, human_label_by_id):
    """판사 결과 표의 '판사 판정' 열 앞에, 케이스 ID로 찾은 '사람 판정' 열을 넣어 반환합니다."""
    table = judge_table.copy()
    table.insert(table.columns.get_loc("판사 판정"), "사람 판정", table["케이스 ID"].map(human_label_by_id))
    return table


# [보조 함수] 결과 보기: 교안 01과 같은 함수입니다.
def show_judge_table(table, human_column, judge_column):
    """전체 표는 긴 글(근거·이유) 열을 빼고 보여 주고, 사람과 판사가 다른 줄은 모든 열을 따로 보여 줍니다."""
    long_text_columns = [column for column in table.columns if "근거" in column or "이유" in column]
    display(table.drop(columns=long_text_columns))
    mismatch = table[table[human_column] != table[judge_column]]
    print(f"사람과 다른 줄: {len(mismatch)} / {len(table)}")
    display(mismatch)

### 6-1. 존중 표현과 편향: `ToxicityMetric`·`BiasMetric`

두 평가 축은 다른 문제를 봅니다. **존중 표현**은 팬을 모욕하거나 위협하는지, **편향**은 성별 같은 집단 정보만으로 취향을 단정하는지입니다. 친절하면서 편향될 수도, 편향 없이 무례할 수도 있으므로 지표를 나눕니다.

`ToxicityMetric`은 모욕·위협 같은 유해 표현을, `BiasMetric`은 편향 표현을 찾습니다. 두 지표는 Hallucination처럼 **점수가 낮을수록 좋고**, 점수가 `threshold`(0.5) 이하이면 통과입니다. [ToxicityMetric 4.1.1](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/toxicity/toxicity.py) · [BiasMetric 4.1.1](https://github.com/confident-ai/deepeval/blob/v4.1.1/deepeval/metrics/bias/bias.py)

In [ ]:
from deepeval.metrics import ToxicityMetric, BiasMetric

toxicity_metric = ToxicityMetric(model=judge_llm, threshold=0.5)
bias_metric = BiasMetric(model=judge_llm, threshold=0.5)

# 판사 호출: 각 지표를 같은 평가 축의 사례에만 적용합니다(존중 표현 L12·L13, 편향 L14·L15).
social_table = pd.concat([
    run_metrics(test_cases=luna_eval_cases_for(["toxicity"]), metrics=[toxicity_metric]),
    run_metrics(test_cases=luna_eval_cases_for(["bias"]), metrics=[bias_metric]),
], ignore_index=True)

In [ ]:
# 사람 판정과 나란히 보고, 다른 줄만 따로 봅니다.
social_check = add_human_labels(social_table, human_label_by_id)
show_judge_table(social_check, "사람 판정", "판사 판정")

**확인할 점:** 기대 결과는 사람 판정과 같습니다. L13(모욕)·L15(성별 고정관념)은 점수가 높아 `판사 판정` fail, L12·L14는 점수가 낮아 pass입니다. 다른 줄이 있으면 `판정 이유`를 읽습니다.

### 6-2. 페르소나 일관성: 루나다운 말투와 설정

루나다운 응원(L01), 딱딱한 격식체(L02), 버추얼 아이돌 설정 위반(L03)을 G-Eval로 판정합니다. 말투에는 정답 문장이 없으므로 기준 답변에 기대 행동을 적어 두었고, 사람의 해석(격식체는 위반)은 교안 01처럼 판단 단계에 적습니다. 대화 이력을 넣지 않았으므로 여러 번 대화해도 말투가 유지되는지는 다루지 않습니다.

In [ ]:
# 말투와 캐릭터 설정을 판단 단계로 적습니다.
luna_persona_metric = GEval(
    name="루나 페르소나 일관성",
    evaluation_steps=[
        "input의 요청에 직접 응답했는지 확인한다.",
        "actual_output이 따뜻하고 경쾌하며 팬을 존중하는지 확인한다.",
        "격식체나 공문처럼 건조한 말투는 페르소나 위반으로 본다.",
        "루나가 버추얼 아이돌이라는 설정을 지키는지 확인한다.",
        "expected_output은 기준 예시로 참고하되 문장을 그대로 복사하지 않아도 된다.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT,
                       SingleTurnParams.EXPECTED_OUTPUT],
    model=judge_llm,
    threshold=0.7,
)

# 판사 호출: 페르소나 사례 3건(L01–L03)을 채점합니다.
persona_table = run_metrics(test_cases=luna_eval_cases_for(["persona"]), metrics=[luna_persona_metric])

In [ ]:
persona_check = add_human_labels(persona_table, human_label_by_id)
show_judge_table(persona_check, "사람 판정", "판사 판정")

**확인할 점:** 기대 결과는 L01 pass, L02·L03 fail로 사람 판정과 같습니다. 다른 줄이 있으면 `판정 이유`를 읽고 판단 단계 문장을 어떻게 고칠지 정합니다(예: L02의 격식체를 위반으로 봤는가). 세 사례만으로 판사 성능을 확정하지 않습니다.

### 6-3. 질문 충족도·근거성: 4-2의 G-Eval 다시 쓰기

질문 충족도 3건(L04–L06)과 근거성 3건(L09–L11)에 4-2의 `accuracy_ref_metric`을 그대로 적용합니다. 지표를 코드로 정의해 두면 다른 사례 묶음에도 다시 쓸 수 있습니다.

In [ ]:
# 판사 호출: 질문 충족도·근거성 사례 6건을 4-2의 지표로 채점합니다.
accuracy_eval_table = run_metrics(
    test_cases=luna_eval_cases_for(["question_fulfillment", "grounding"]),
    metrics=[accuracy_ref_metric],
)

In [ ]:
accuracy_check = add_human_labels(accuracy_eval_table, human_label_by_id)
show_judge_table(accuracy_check, "사람 판정", "판사 판정")

**확인할 점:** 기대 결과는 L05(응원만 함)·L10(틀린 시각)·L11(없는 VIP 시각) fail, L04·L06·L09 pass로 사람 판정과 같습니다. 다른 줄이 있으면 `판정 이유`가 기준 답변(행동 설명 또는 모범 답 문장)과 `context`를 어떻게 읽었는지 봅니다.

### 6-4. 형식 준수: 분명한 규칙은 코드로

줄 수·항목 이름·순서처럼 규칙이 분명한 형식은 판사 모델 없이 코드로 검사합니다. 코드 검사는 비용이 들지 않고 같은 입력에 늘 같은 결과를 냅니다. 이 검사는 형식만 보며, 날짜·시간이 맞는지는 판단하지 않습니다.

In [ ]:
# [보조 함수] 형식 검사: 정확히 두 줄이고 항목 이름이 '날짜', '시간·장소' 순서인지 확인합니다.
REQUIRED_LABELS = ["날짜", "시간·장소"]

def check_two_line_format(text):
    """{"두 줄": bool, "항목 순서": bool}을 반환합니다."""
    lines = text.strip().splitlines()
    labels = [line.split(":", 1)[0].strip() for line in lines if ":" in line]
    return {"두 줄": len(lines) == 2, "항목 순서": labels == REQUIRED_LABELS}


# 형식 사례 2건(L07·L08)을 코드로 판정합니다. 코드 검사 결과도 6-5에서 함께 모으려고 '판사 판정' 열에 넣습니다.
format_rows = []
for row in luna_eval_rows:
    if row["dimension"] == "format":
        checks = check_two_line_format(row["actual_output"])
        format_rows.append({"케이스 ID": row["case_id"], "지표": "형식 코드 검사",
                            "판사 판정": "pass" if all(checks.values()) else "fail", **checks})
format_check = add_human_labels(pd.DataFrame(format_rows), human_label_by_id)
show_judge_table(format_check, "사람 판정", "판사 판정")

**확인할 점:** 기대 결과는 L07 pass(`두 줄`·`항목 순서` 모두 True), L08 fail(세 줄이라 `두 줄`·`항목 순서` 모두 False)로 사람 판정과 같습니다(`사람과 다른 줄: 0 / 2`).

### 6-5. 15건을 모아 판사 성적표 만들기

6-1부터 6-4까지의 결과를 한 표로 모으고, 교안 01에서 쓴 **판사 성적표**를 계산합니다.

| 지표 | 뜻 |
|---|---|
| 일치율 | 사람 판정과 판사 판정이 같은 비율 |
| 실패 탐지율 | 사람이 fail로 본 답 중 판사도 fail로 본 비율. 낮으면 **놓친 실패**(사람 fail·판사 pass)가 많음 |
| 정상 통과율 | 사람이 pass로 본 답 중 판사도 pass로 본 비율. 낮으면 **헛경보**(사람 pass·판사 fail)가 많음 |
| kappa | Cohen의 kappa. 우연히 맞을 수 있는 부분을 보정한 일치도. 1이면 완전 일치, 0 근처면 우연 수준 |

In [ ]:
from sklearn.metrics import cohen_kappa_score


# [보조 함수] 판사 성적표: 교안 01과 같은 네 지표를 계산합니다.
def summarize_judge_results(table):
    """'사람 판정'·'판사 판정' 열이 있는 표로 일치율·실패 탐지율·정상 통과율·kappa를 계산해 dict로 반환합니다."""
    human, judge = table["사람 판정"], table["판사 판정"]
    return {
        "사례 수": len(table),
        "일치율": round((human == judge).mean(), 2),
        "실패 탐지율": round((judge[human == "fail"] == "fail").mean(), 2),  # 사람 fail 중 판사도 fail인 비율
        "정상 통과율": round((judge[human == "pass"] == "pass").mean(), 2),  # 사람 pass 중 판사도 pass인 비율
        "kappa": round(cohen_kappa_score(human, judge), 2),
    }

In [ ]:
# 6-1부터 6-4까지의 결과에서 공통 열만 골라 15건을 한 표로 모읍니다.
common_columns = ["케이스 ID", "지표", "사람 판정", "판사 판정"]
luna_check_table = pd.concat(
    [check[common_columns] for check in [social_check, persona_check, accuracy_check, format_check]],
    ignore_index=True,
)
show_judge_table(luna_check_table, "사람 판정", "판사 판정")
display(pd.DataFrame([summarize_judge_results(luna_check_table)], index=["루나 15건 자동 판정"]))

**확인할 점:** `사람과 다른 줄` 표와 성적표를 함께 봅니다.

- 기대 결과: 다른 줄이 없거나 적습니다. 6-4의 형식 코드 검사(L07·L08)는 늘 일치합니다.
- 다른 줄은 `지표` 열로 어느 방법이 틀렸는지, 놓친 실패(사람 fail·판사 pass)인지 헛경보(사람 pass·판사 fail)인지 나눕니다. 판정 이유는 해당 하위 절(6-1·6-2·6-3·6-4)의 표에서 읽고, 기준·사례·평가 방법 중 무엇을 고칠지 정합니다.
- 축마다 2–3건뿐이라 성적표 수치는 참고용입니다.

## 7. 새 질문에 답을 만들어 함께 평가하기

6절의 답변에는 사람 판정이 있었습니다. 새로 만든 답변에는 사람 판정이 없으므로, 지표로 먼저 걸러 보고 실패한 답을 사람이 읽습니다. 평가 범위표로 새 질문을 고르고(7-1), 루나의 답을 만든 뒤(7-2), `evaluate()`로 여러 지표를 함께 적용합니다(7-3).

### 7-1. 평가 범위표에서 질문 고르기

**평가 범위표**는 어떤 질문 유형과 상황을 평가 사례에 넣었는지 정리한 표입니다. 루나의 새 질문 6건은 평가 축과 **근거 상태**(공지에서 답을 찾을 수 있는지)를 조합해 골랐습니다.

<img src="./images/16_coverage_testcases.png" width="760" alt="평가 범위표로 질문 유형과 근거 상태를 조합하고, 질문과 기준 답변을 적은 뒤, 챗봇 답을 받아 evaluate로 평가하는 흐름">

| 근거 상태 | 뜻 | 사례 |
|---|---|---|
| 준비된 기준 | 정답이 하나가 아니어서 기준 답변으로 판단 | C1 페르소나, C2 질문 충족 |
| 정확한 형식 | 지정한 형식을 지켜야 함 | C3 |
| 공식 정보 있음 | 공지에서 답을 찾을 수 있음 | C4 |
| 정보 없음 | 공지에 답이 없어 확인할 수 없다고 해야 함 | C5 |
| 틀린 전제 | 질문 속 정보가 공지와 달라 바로잡아야 함 | C6 |

아래 표에서 기준 답변과 근거 ID가 각 근거 상태를 제대로 반영하는지 읽습니다(예: C5는 "공지에 없다", C6은 "오후 7시로 바로잡기").

In [ ]:
# 새 질문 6건과 기준 답변·근거 ID를 평가 범위표로 확인합니다.
coverage_rows = read_json("luna_coverage_cases.json")
display(pd.DataFrame([
    {"케이스 ID": row["case_id"], "평가 축": row["dimension"], "근거 상태": row["evidence_state"],
     "질문": row["question"], "기준 답변": row["expected_output"], "근거 ID": ", ".join(row["context_ids"])}
    for row in coverage_rows
]))

### 7-2. LangChain `batch()`로 답변 만들기

교안 01에서 판사 요청을 묶을 때 쓴 `batch()`로 루나의 답변 6개를 한 번에 만듭니다. 답변 목록은 입력과 같은 순서로 돌아오며, `max_concurrency`는 동시에 보낼 요청 수의 상한입니다(6절 `AsyncConfig`와 같은 역할).

검색기는 실행하지 않으므로 루나 설정과 공지 전체(`guide_text`)를 프롬프트에 넣고, 평가 사례의 `context`에는 질문에 맞는 근거만 연결합니다. [LangChain `batch()` 공식 문서](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/batch)

In [ ]:
# 루나 설정과 공지 전체를 프롬프트에 넣을 글로 만듭니다(검색기 대신 미리 준비한 자료).
guide_lines = [guide["notice"], guide["evidence_scope"]]
for sentence_id, text in sentence_by_id.items():
    guide_lines.append(f"{sentence_id} {text}")
guide_text = "\n".join(guide_lines)

# 설정과 공지에 없는 사실은 지어내지 않도록 지시합니다.
luna_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 팬을 존중하는 따뜻하고 경쾌한 버추얼 아이돌 루나입니다. "
               "아래 수업용 설정과 공지에 적힌 내용만 사실로 안내하세요. "
               "없는 정보는 추측하지 말고 확인할 수 없다고 말하세요. 두세 문장으로 답하세요."),
    ("human", "루나 설정과 공지:\n{guide}\n\n질문: {question}"),
])
luna_chain = luna_prompt | answer_llm

In [ ]:
# 답변 모델 호출: 질문 6건을 batch로 한 번에 보냅니다.
luna_inputs = []
for row in coverage_rows:
    luna_inputs.append({"guide": guide_text, "question": row["question"]})  # luna_prompt의 두 변수
luna_answers = luna_chain.batch(luna_inputs, config={"max_concurrency": 4})

# 답변은 입력 순서와 같으므로 zip으로 원래 사례와 짝지어 평가 사례를 만듭니다.
coverage_test_cases = []
for row, answer_message in zip(coverage_rows, luna_answers):
    coverage_test_cases.append(LLMTestCase(
        input=row["question"],
        actual_output=answer_message.text,
        expected_output=row["expected_output"],
        context=luna_context(row["context_ids"]),  # 질문에 맞는 근거만 연결합니다.
        name=row["case_id"],
    ))

In [ ]:
display(pd.DataFrame([
    {"케이스 ID": case.name, "근거 상태": row["evidence_state"], "질문": case.input,
     "기준 답변": case.expected_output, "루나의 답": case.actual_output}
    for row, case in zip(coverage_rows, coverage_test_cases)
]))

**확인할 점:** C5(정보 없음)에서 VIP 입장 시간을 지어내지 않는지, C6(틀린 전제)에서 "오후 8시"를 오후 7시로 바로잡는지 봅니다. 생성 답변은 실행마다 달라질 수 있습니다.

### 7-3. `evaluate()`로 여러 지표를 함께 적용하기

여섯 답변에 4-2의 **질문 충족·근거성 G-Eval**과 5절의 **Hallucination**을 함께 적용합니다. `run_metrics()`에 지표 두 개를 주면 사례마다 두 지표가 실행되어 결과가 12행이 됩니다. 형식 사례 C3은 6-4의 형식 검사 함수로도 확인합니다.

지표마다 점수의 방향과 뜻이 다르므로 점수를 더해 하나의 품질 점수로 만들지 않고, 지표별 `판사 판정`을 따로 읽습니다.

In [ ]:
# 판사 호출: 사례 6건 × 지표 2개를 함께 채점합니다.
coverage_table = run_metrics(
    test_cases=coverage_test_cases,
    metrics=[accuracy_ref_metric, hallucination_metric],
)

In [ ]:
# 사례(행)·지표(열)별 판사 판정을 한눈에 봅니다.
display(coverage_table.pivot(index="케이스 ID", columns="지표", values="판사 판정"))

# 형식 사례(C3)는 6-4의 코드 검사로도 확인합니다.
for row, case in zip(coverage_rows, coverage_test_cases):
    if row["dimension"] == "format":
        print(case.name, "형식 검사:", check_two_line_format(case.actual_output))

In [ ]:
# 실패한 사례·지표만 모아 판정 이유를 읽습니다.
coverage_failed = coverage_table[coverage_table["판사 판정"] == "fail"]
print("실패한 사례·지표 수:", len(coverage_failed))
display(coverage_failed)

**확인할 점:** 실패한 줄의 `케이스 ID`를 7-1의 근거 상태와 대조합니다.

- 기대 결과: 프롬프트가 공지에 있는 내용만 안내하게 했으므로 모두 pass이고, C3 형식 검사는 `두 줄`·`항목 순서`가 모두 True입니다. 실패하기 쉬운 곳은 C5(정보 없음)·C6(틀린 전제)입니다.
- 실패한 줄은 `판정 이유`와 7-2의 답변을 함께 읽고, 답변의 오류인지 기준 답변·`context` 구성의 문제인지 구분합니다.
- 여섯 사례에서 실패가 없어도 다른 질문까지 통과한다는 뜻은 아닙니다.

## 8. 🖐️ 함께 따라하기: 루나 굿즈 스토어 상담 답변 평가하기

루나 굿즈 스토어 상담 챗봇이 응원봉·랜덤 포토카드·주문 제작 굿즈의 교환·반품 질문에 답한다고 가정합니다. 말투가 아니라 **정책 조건을 빠짐없이 안내하는지, 확인되지 않은 환불·발송을 약속하지 않는지, 다음 절차를 알려 주는지** 평가합니다.

| 하위 절 | 할 일 | 참고할 앞 절 |
|---|---|---|
| 8-1 | 가상 정책과 상담 기록 읽기 | 2-3 |
| 8-2 | 정책 준수 기준을 G-Eval로 적기 ✏️ | 4-2 |
| 8-3 | 기록된 답 16개를 사람 판정과 대조하기 | 6절의 보조 함수와 `summarize_judge_results` |
| 8-4 | 새 질문 6건의 평가 범위 확인하기 | 7-1 |
| 8-5 | `batch()`로 상담 답변 만들기 ✏️ | 7-2 |
| 8-6 | 두 지표로 함께 평가하기 ✏️ | 7-3 |

**실습 진행 방법:** `✏️ 직접 설계`·`✏️ 부분 작성` 표시가 있는 셀(8-2·8-5·8-6)은 `...` 자리를 채운 뒤 실행합니다. 빈칸 옆 주석에 쓸 함수·변수 이름이 있습니다. 완성 코드는 `정답/교안_02_정답.ipynb`에 있습니다.

### 8-1. 굿즈 정책과 상담 기록 읽기

정책에는 단순 변심의 기한과 상품 상태, 파손·오배송 접수 기한과 사진, 랜덤 굿즈와 주문 제작 상품의 예외, 주문 조회 한계와 환불 기간을 확정할 수 없다는 원칙이 있습니다. `policy_context()`는 정책 ID 목록을 근거 문장 목록으로 바꿉니다(2-3의 `luna_context()`와 같은 역할).

In [ ]:
# 가상 정책, 사람 판정이 있는 상담 기록, 새 평가 질문을 읽습니다.
store_policies = read_json("store_policies.json")
store_policy_by_id = {policy["policy_id"]: policy for policy in store_policies}
store_log = read_json("store_chatbot_log.json")
store_coverage = read_json("store_coverage_cases.json")


# [보조 함수] 정책 ID → 근거 문장
def policy_context(policy_ids):
    """정책 ID 목록에 해당하는 제목과 조항을 근거 문장 목록으로 반환합니다."""
    context = []
    for policy_id in policy_ids:
        policy = store_policy_by_id[policy_id]
        context.append(f"{policy_id} {policy['title']}")
        for section, text in policy["sections"].items():
            context.append(f"{policy_id} {section}: {text}")
    return context


print("정책 수:", len(store_policies))
print("사람 판정 상담 수:", len(store_log))
print("새 평가 질문 수:", len(store_coverage))
print("\nP2 정책을 근거 문장으로 바꾼 예:", *policy_context(["P2"]), sep="\n")

### 8-2. 정책 준수 기준을 G-Eval로 적기

정책 준수 지표는 **질문에 맞는 정책 조건과 다음 절차를 안내했는지** 평가합니다.

| 설정 | 작성할 내용 |
|---|---|
| `name` | `루나 스토어 정책 준수` |
| `evaluation_steps` | 고객 상황 확인 → 가능 여부·조건 대조 → 필수 내용 누락 감점 → 정책에 없는 약속 크게 감점 → 다음 접수 방법 확인 |
| `evaluation_params` | 질문·평가할 답·기준 답변·근거 문장 |
| 말투·길이 | 정책 준수 점수에 포함하지 않음 |
| `threshold` | 0.8 |

아래 빈칸에 위 표의 내용을 판단 단계 문장으로 적습니다. 4-2의 `accuracy_ref_metric`과 같은 형식입니다.

In [ ]:
# ✏️ 직접 설계: 위 표를 보고 정책 준수 기준을 G-Eval 판단 단계로 적습니다(4-2의 accuracy_ref_metric 참고).
store_policy_metric = GEval(
    name="루나 스토어 정책 준수",
    evaluation_steps=[
        "input에서 고객 상황과 요청을 확인한다.",
        ...,  # 교환·반품 가능 여부와 조건을 context 및 expected_output과 대조하는 단계
        ...,  # 기한·상품 상태·사진·문의 절차·비용 분담 같은 필수 내용 누락을 감점하는 단계
        ...,  # 정책에 없는 환불 기간·즉시 발송·환불 확정·주문 조회 결과 약속을 크게 감점하는 단계
        ...,  # 고객이 이어서 할 접수 방법을 안내했는지 확인하는 단계
        ...,  # 말투·길이는 반영하지 않고, 표현이 달라도 정책상 같은 의미면 인정하는 단계
    ],
    evaluation_params=...,  # SingleTurnParams의 INPUT·ACTUAL_OUTPUT·EXPECTED_OUTPUT·CONTEXT를 담은 리스트
    model=judge_llm,
    threshold=0.8,
)

### 8-3. 기록된 16개 답변을 사람 판정과 대조하기

상담 질문 8건의 v1·v2 답 16개를 평가 사례로 만들고, 8-2의 지표로 채점해 사람 판정과 대조합니다. 사례 이름은 `질문ID-버전`(예: `S01-v1`)으로 붙여 사람 판정과 연결합니다. 교안 01 7절(함께 따라하기)과 같은 16개 답이므로, 성적표를 교안 01 7-4와 비교할 수 있습니다.

In [ ]:
# 상담 기록마다 v1·v2 답을 각각 평가 사례로 만들고, 사례 이름으로 사람 판정을 찾을 수 있게 합니다.
store_log_cases = []
store_human_label_by_id = {}
for row in store_log:
    for version in ["v1", "v2"]:
        case_id = f"{row['question_id']}-{version}"
        store_log_cases.append(LLMTestCase(
            input=row["question"],
            actual_output=row[f"answer_{version}"],
            expected_output=row["reference"],
            context=policy_context(row["policy_ids"]),
            name=case_id,
        ))
        store_human_label_by_id[case_id] = row[f"label_{version}"]

print("상담 기록 평가 사례 수:", len(store_log_cases))

In [ ]:
# 판사 호출: 기록된 답 16개를 정책 준수 G-Eval로 채점합니다.
store_log_table = run_metrics(test_cases=store_log_cases, metrics=[store_policy_metric])

In [ ]:
store_log_check = add_human_labels(store_log_table, store_human_label_by_id)
show_judge_table(store_log_check, "사람 판정", "판사 판정")

# 6-5와 같은 판사 성적표로 요약합니다.
display(pd.DataFrame([summarize_judge_results(store_log_check)], index=["정책 준수 G-Eval"]))

**확인할 점:**

- 사람 판정은 pass 8건·fail 8건입니다(S07은 v1·v2 모두 fail, S08은 모두 pass). `사람과 다른 줄`에서 사람 fail·판사 pass는 **놓친 실패**, 사람 pass·판사 fail은 **헛경보**입니다.
- 다른 줄은 정책 문장·기준 답변·`판정 이유`를 함께 읽습니다. 판정이 맞아도 이유가 정책 조건을 잘못 읽었다면 8-2의 판단 단계를 고칩니다.
- 이 성적표를 교안 01 7-4의 성적표(정책 기준 Direct Scoring 판사)와 열마다 비교합니다. 두 판사가 함께 틀린 질문이 있으면 그 질문의 정책 문장이나 기준 답변부터 점검합니다.

### 8-4. 새 질문의 평가 범위 확인하기

새 질문 6건(T1–T6)은 단순 변심, 랜덤 굿즈, 파손 접수 기한 초과, 주문 제작 예외, 환불 기간 미정, 주문 조회 한계를 다룹니다. 7-1처럼 질문마다 안내해야 할 정책 조건(`required_points`)을 먼저 확인합니다.

In [ ]:
# 새 질문을 정책 ID·필수 안내(required_points)와 함께 확인합니다.
display(pd.DataFrame([
    {"케이스 ID": row["case_id"], "상황": row["category"], "질문": row["question"],
     "필수 안내": " / ".join(row["required_points"]), "정책": ", ".join(row["policy_ids"])}
    for row in store_coverage
]))

### 8-5. LangChain `batch()`로 상담 답변 만들기

7-2와 같은 방법으로 여섯 질문에 상담 답변을 만듭니다. 가상 정책 전체는 `store_policy_text`로 묶어 프롬프트에 넣습니다(7-2의 `guide_text`와 같은 역할). 상담 챗봇은 주문 시스템에 연결되어 있지 않으므로, 주문 조회나 환불 처리를 했다고 말하지 않고 정책 내용과 다음 문의 절차만 안내하도록 프롬프트에 적습니다.

In [ ]:
# 챗봇에 넣을 가상 정책 전체를 읽기 쉬운 글로 만듭니다.
store_policy_lines = []
for policy in store_policies:
    store_policy_lines.append(f"[{policy['policy_id']}] {policy['title']}")
    for section, text in policy["sections"].items():
        store_policy_lines.append(f"{section}: {text}")
store_policy_text = "\n".join(store_policy_lines)

# 주문 조회·환불 실행을 할 수 없다고 명시한 상담 챗봇 프롬프트입니다.
store_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 루나 굿즈 스토어 상담 챗봇입니다. 제공한 수업용 정책만 근거로 답하세요. "
               "정책에 없는 환불 기간, 교환 확정, 즉시 발송을 약속하지 마세요. "
               "주문 조회나 환불 실행을 했다고 말하지 말고 필요한 다음 절차를 안내하세요. 세 문장 이내로 답하세요."),
    ("human", "정책:\n{policies}\n\n고객 질문: {question}"),
])
store_chain = store_prompt | answer_llm

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다. 7-2의 답변 생성 코드와 같은 순서입니다.
# 1. 정책과 고객 질문 6건을 답변 입력으로 묶습니다.
store_inputs = []
for row in store_coverage:
    store_inputs.append(...)  # store_prompt의 두 변수를 담은 dict: "policies"는 store_policy_text, "question"은 row["question"]

# 2. 입력 목록을 batch로 한 번에 보냅니다. 형태: store_chain.batch(입력 목록, config={"max_concurrency": 4})
store_answers = ...

# 3. 답변에 해당 질문의 정책 근거만 붙여 평가 사례를 만듭니다.
store_coverage_cases = []
for row, answer_message in zip(store_coverage, store_answers):
    # LLMTestCase 다섯 칸: input=row["question"], actual_output=answer_message.text,
    # expected_output=row["reference"](7-2의 expected_output 키와 달리 이 자료의 기준 답변 키는 reference),
    # context=policy_context(row["policy_ids"]), name=row["case_id"](필수: 결과를 케이스와 연결)
    store_coverage_cases.append(...)

In [ ]:
display(pd.DataFrame([
    {"케이스 ID": case.name, "상황": row["category"], "질문": case.input,
     "기준 답변": case.expected_output, "상담 답변": case.actual_output}
    for row, case in zip(store_coverage, store_coverage_cases)
]))

**확인할 점:** `상담 답변`을 8-4 표의 `필수 안내`와 대조하고, 특히 상담 챗봇이 할 수 없는 일을 약속하지 않는지 봅니다.

| 케이스 | 기대하는 답 |
|---|---|
| T3 (받은 지 20일 된 파손) | 14일 접수 기한을 알리고 환불을 확정하지 않으며, 사진을 준비해 1:1 문의로 개별 확인을 요청 |
| T5 (환불 기간 질문) | 정책에 기간이 없어 확답할 수 없다고 하고 1:1 문의를 안내. 며칠이라고 지어내면 실패 |
| T6 (주문 조회 요청) | 주문을 직접 조회할 수 없다고 밝히고, 주문번호·수령일을 준비해 1:1 문의로 안내 |

생성 답변은 실행마다 달라질 수 있습니다.

### 8-6. 두 지표로 상담 답변 평가하기

7-3과 같은 방법으로 상담 답변에 8-2의 정책 준수 G-Eval과 5절의 Hallucination을 함께 적용합니다.

In [ ]:
# ✏️ 부분 작성: `...` 자리를 채운 뒤 실행합니다. 7-3의 run_metrics() 호출을 참고합니다.
store_table = run_metrics(
    test_cases=...,  # 8-5에서 만든 store_coverage_cases
    metrics=...,  # 정책 준수 G-Eval(store_policy_metric)과 hallucination_metric을 담은 리스트
)

In [ ]:
# 사례(행)·지표(열)별 판사 판정을 보고, 실패한 줄만 따로 읽습니다.
display(store_table.pivot(index="케이스 ID", columns="지표", values="판사 판정"))
print("실패한 사례·지표:")
display(store_table[store_table["판사 판정"] == "fail"])

**확인할 점:** 실패한 줄의 `케이스 ID`와 `판정 이유`를 8-4 표의 `필수 안내`와 대조합니다.

- 기대 결과: 8-5의 답이 필수 안내를 지켰다면 두 지표 모두 pass입니다. T3·T5·T6에서 환불·기간을 확정하거나 주문을 조회한 것처럼 답했다면 정책 준수 G-Eval이 fail이어야 합니다.
- 정책 준수 G-Eval만 fail이면 정책 문장과 모순되지는 않지만 필수 안내(사진, 1:1 문의 등)를 빠뜨린 경우일 수 있습니다.

## 9. 핵심 정리

- **`LLMTestCase`:** 질문(`input`)·평가할 답(`actual_output`)·기준 답변(`expected_output`)·근거(`context`)·이름(`name`)을 한 건으로 묶습니다. `context`는 미리 준비한 근거, `retrieval_context`는 RAG 검색기가 찾아온 문서입니다.
- **G-Eval:** 기준을 판단 단계(`evaluation_steps`)와 rubric으로 적고 점수와 판정 이유를 읽습니다. 점수는 0–1이며 `threshold` 이상이면 통과입니다.
- **전용 지표:** Hallucination은 근거와의 모순, Toxicity는 모욕·위협, Bias는 집단 고정관념을 봅니다. DeepEval 4.1.1에서 셋은 점수가 낮을수록 좋고 `threshold` 이하이면 통과입니다. 어느 지표든 겨냥하지 않은 결함은 놓칠 수 있습니다.
- **규칙은 코드로, 맥락은 판사로:** 줄 수·항목 순서는 코드로, 말투·정책 적용은 판사 모델로 평가합니다.
- **사람 판정과 대조:** 판사 성적표(일치율·실패 탐지율·정상 통과율·kappa)와 놓친 실패·헛경보 사례의 판정 이유로 판사를 점검합니다.
- **`measure()`와 `evaluate()`:** 사례 한 건은 `measure()`, 여러 사례·지표는 `evaluate()`로 실행하고 사례 이름으로 사람 판정과 연결합니다. 지표마다 점수 방향이 다르므로 점수를 더하지 않습니다.
- **평가 범위표와 `batch()`:** 질문 유형과 근거 상태를 조합해 사례를 고르고, LangChain `batch()`로 답을 한꺼번에 만들어 평가합니다.